1.Load dataset

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "/multi_platform_social_sentiment_evolution.csv"

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("DATASET LOADED")
print(df.shape)
print("=" * 60)

print(df.head())

DATASET LOADED
(150000, 31)
             post_id   platform            timestamp        date  hour_of_day  \
0  TIK20250419000000     TikTok  2025-04-19 01:56:55  2025-04-19            1   
1  TWI20250419000001    Twitter  2025-04-19 05:34:09  2025-04-19            5   
2  INS20250419000002  Instagram  2025-04-19 06:33:36  2025-04-19            6   
3  INS20250419000003  Instagram  2025-04-19 06:42:16  2025-04-19            6   
4  RED20250419000004     Reddit  2025-04-19 06:46:49  2025-04-19            6   

   day_of_week  is_weekend      user_id  followers  account_age_days  ...  \
0            5           1  user_426711        137               306  ...   
1            5           1  user_221610       1974              2310  ...   
2            5           1    user_7998       6471              1990  ...   
3            5           1  user_313440       1366              2057  ...   
4            5           1   user_23343       1349              1445  ...   

   shares comments vie

2.  FEATURE ENGINEERING

 3.1. TIMESTAMP CONVERSION

In [2]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

2.2. CYCLICAL TIME ENCODING

In [3]:
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour_of_day"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour_of_day"] / 24
)

# Day encoding
df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

2.3. LOG TRANSFORM FOLLOWERS

In [4]:
df["followers_log"] = np.log1p(df["followers"])

2.4. CONTENT FLAGS

In [5]:
# Has media or not
df["has_media"] = np.where(
    df["media_type"] == "Text",
    0,
    1
)

# Heavy hashtag usage
df["many_hashtags"] = np.where(
    df["num_hashtags"] >= 5,
    1,
    0
)

# Long content flag
df["long_content"] = np.where(
    df["content_length"] > df["content_length"].median(),
    1,
    0
)


2.5. SENTIMENT INTENSITY

In [6]:
sentiment_cols = [

    "sentiment_positive",
    "sentiment_negative",
    "sentiment_neutral"
]

for col in sentiment_cols:

    if col in df.columns:

        # Force probabilities into [0,1]
        df[col] = df[col].clip(0, 1)

In [7]:
df["sentiment_intensity"] = df[
    [
        "sentiment_positive",
        "sentiment_negative",
        "sentiment_neutral"
    ]
].max(axis=1)


3.6. TOXIC CONTENT FLAG

In [8]:
df["high_toxicity"] = np.where(
    df["toxicity_score"] >= 70,
    1,
    0
)

3.7. VERIFIED + LARGE ACCOUNT

In [9]:
followers_threshold = df["followers"].quantile(0.90)

df["large_account"] = np.where(
    df["followers"] >= followers_threshold,
    1,
    0
)

3.7 CHECK NEW FEATURES

In [10]:
new_features = [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "followers_log",
    "has_media",
    "many_hashtags",
    "long_content",
    "sentiment_intensity",
    "high_toxicity",
    "large_account"
]

print("\nNew engineered features:")
print(new_features)


New engineered features:
['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'followers_log', 'has_media', 'many_hashtags', 'long_content', 'sentiment_intensity', 'high_toxicity', 'large_account']


4.DEFINE TARGET (POPULARITY LABEL)

In [11]:
platform_dfs = []

platforms = df["platform"].unique()
for platform in platforms:
    # FILTER PLATFORM
    temp_df = df[
        df["platform"] == platform
    ].copy()
    # TOP 20% = POPULAR
    threshold = temp_df[
        "total_engagement"
    ].quantile(0.80)

    temp_df["popularity"] = np.where(
        temp_df["total_engagement"] >= threshold,
        1,
        0
    )
    platform_dfs.append(temp_df)

df = pd.concat(
    platform_dfs,
    ignore_index=True
)
print("=" * 60)
print("POPULARITY DISTRIBUTION")
print("=" * 60)

print(df["popularity"].value_counts())

print("\nRatio:")
print(df["popularity"].value_counts(normalize=True))


POPULARITY DISTRIBUTION
popularity
0    119771
1     30229
Name: count, dtype: int64

Ratio:
popularity
0    0.798473
1    0.201527
Name: proportion, dtype: float64


In [12]:
print(df.columns.tolist())

['post_id', 'platform', 'timestamp', 'date', 'hour_of_day', 'day_of_week', 'is_weekend', 'user_id', 'followers', 'account_age_days', 'verified', 'topic', 'language', 'content_length', 'media_type', 'num_hashtags', 'sentiment_category', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'likes', 'shares', 'comments', 'views', 'total_engagement', 'engagement_rate_per_1k_followers', 'hours_since_post', 'viral_coefficient', 'cross_platform_spread', 'toxicity_score', 'location', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'followers_log', 'has_media', 'many_hashtags', 'long_content', 'sentiment_intensity', 'high_toxicity', 'large_account', 'popularity']


5.CREATE OUTPUT FOLDERS

In [13]:
from pathlib import Path
base_dir = Path("processed_data")

graph_dir = base_dir / "graph_data"
ml_dir = base_dir / "ml_data"

graph_dir.mkdir(parents=True, exist_ok=True)
ml_dir.mkdir(parents=True, exist_ok=True)

5.REMOVE LEAKAGE COLUMNS

In [14]:

LEAKAGE_COLS = [
    # raw datetime
    "timestamp",
    "date",

    # direct engagement metrics
    "likes",
    "shares",
    "comments",
    "views",
    "total_engagement",

    # derived engagement metrics
    "engagement_rate_per_1k_followers",
    "viral_coefficient",

    # virality outcome
    "cross_platform_spread",

    # future-time leakage
    "hours_since_post"
]

# -----------------------------------------------------
# REMOVE ONLY EXISTING COLUMNS
# -----------------------------------------------------

existing_leakage_cols = [

    col for col in LEAKAGE_COLS
    if col in df.columns
]

df = df.drop(
    columns=existing_leakage_cols
)

print("=" * 60)
print("LEAKAGE COLUMNS REMOVED")
print("=" * 60)

print(existing_leakage_cols)

print("\nCurrent Shape:")
print(df.shape)

LEAKAGE COLUMNS REMOVED
['timestamp', 'date', 'likes', 'shares', 'comments', 'views', 'total_engagement', 'engagement_rate_per_1k_followers', 'viral_coefficient', 'cross_platform_spread', 'hours_since_post']

Current Shape:
(150000, 32)


6. SAVE GRAPH DATASET SPLIT BY PLATFORM

In [15]:
platform_list = df["platform"].unique()

print(platform_list)

for platform_name in platform_list:
    platform_df = df[
        df["platform"] == platform_name
    ].copy()

    print(f"\nPROCESSING: {platform_name}")
    print("Shape:", platform_df.shape)
    graph_df = platform_df.copy()

    graph_path = graph_dir / f"{platform_name.lower()}_graph.csv"

    graph_df.to_csv(
        graph_path,
        index=False
    )

    print(f"Graph dataset saved: {graph_path}")

<StringArray>
['TikTok', 'Twitter', 'Instagram', 'Reddit', 'Facebook', 'YouTube']
Length: 6, dtype: str

PROCESSING: TikTok
Shape: (11887, 32)
Graph dataset saved: processed_data\graph_data\tiktok_graph.csv

PROCESSING: Twitter
Shape: (44676, 32)
Graph dataset saved: processed_data\graph_data\twitter_graph.csv

PROCESSING: Instagram
Shape: (30195, 32)
Graph dataset saved: processed_data\graph_data\instagram_graph.csv

PROCESSING: Reddit
Shape: (37585, 32)
Graph dataset saved: processed_data\graph_data\reddit_graph.csv

PROCESSING: Facebook
Shape: (7527, 32)
Graph dataset saved: processed_data\graph_data\facebook_graph.csv

PROCESSING: YouTube
Shape: (18130, 32)
Graph dataset saved: processed_data\graph_data\youtube_graph.csv


7.SAVE Ml DATASET

In [16]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

for platform_name in platform_list:

    print("\n" + "=" * 60)
    print(f"PROCESSING: {platform_name}")
    print("=" * 60)
    platform_df = df[
        df["platform"] == platform_name
    ].copy()

    print("Original Shape:", platform_df.shape)
    ml_df = platform_df.copy()
    # CLEANER ML PIPELINE
    remove_cols = [
        "post_id",
        "user_id",
        "timestamp",
        "date",
        "platform",
        "sentiment_category"
    ]
    existing_remove_cols = [

        col for col in remove_cols
        if col in ml_df.columns
    ]

    ml_df = ml_df.drop(
        columns=existing_remove_cols
    )
    if "sentiment_neutral" in ml_df.columns:

        ml_df = ml_df.drop(
            columns=["sentiment_neutral"]
        )

    # ONE-HOT ENCODING
    categorical_cols = [

        "topic",
        "language",
        "media_type",
        "location"
    ]

    existing_categorical_cols = [

        col for col in categorical_cols
        if col in ml_df.columns
    ]

    ml_df = pd.get_dummies(
        ml_df,
        columns=existing_categorical_cols,
        drop_first=True
    )


    bool_cols = ml_df.select_dtypes(
        include="bool"
    ).columns

    ml_df[bool_cols] = ml_df[
        bool_cols
    ].astype(int)

    constant_cols = [

        col for col in ml_df.columns
        if ml_df[col].nunique() <= 1
    ]

    if len(constant_cols) > 0:

        ml_df = ml_df.drop(
            columns=constant_cols
        )

        print("\nRemoved constant columns:")
        print(constant_cols)

    print("\nFinal ML Shape:", ml_df.shape)

    X = ml_df.drop(
        columns=["popularity"]
    )

    y = ml_df["popularity"]

    # TRAIN / TEST SPLIT
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    train_df = X_train.copy()

    train_df["popularity"] = y_train.values

    # CREATE TEST DATAFRAME
    test_df = X_test.copy()

    test_df["popularity"] = y_test.values

    # SAVE TRAIN DATASET
    train_path = ml_dir / f"{platform_name.lower()}_train.csv"

    train_df.to_csv(
        train_path,
        index=False
    )

    # SAVE TEST DATASET

    test_path = ml_dir / f"{platform_name.lower()}_test.csv"

    test_df.to_csv(
        test_path,
        index=False
    )
    # PRINT SUMMARY

    print(f"\nTrain saved: {train_path}")
    print(f"Test saved : {test_path}")

    print("Train Shape:", train_df.shape)
    print("Test Shape :", test_df.shape)


PROCESSING: TikTok
Original Shape: (11887, 32)

Final ML Shape: (11887, 56)

Train saved: processed_data\ml_data\tiktok_train.csv
Test saved : processed_data\ml_data\tiktok_test.csv
Train Shape: (9509, 56)
Test Shape : (2378, 56)

PROCESSING: Twitter
Original Shape: (44676, 32)

Final ML Shape: (44676, 56)

Train saved: processed_data\ml_data\twitter_train.csv
Test saved : processed_data\ml_data\twitter_test.csv
Train Shape: (35740, 56)
Test Shape : (8936, 56)

PROCESSING: Instagram
Original Shape: (30195, 32)

Final ML Shape: (30195, 56)

Train saved: processed_data\ml_data\instagram_train.csv
Test saved : processed_data\ml_data\instagram_test.csv
Train Shape: (24156, 56)
Test Shape : (6039, 56)

PROCESSING: Reddit
Original Shape: (37585, 32)

Final ML Shape: (37585, 56)

Train saved: processed_data\ml_data\reddit_train.csv
Test saved : processed_data\ml_data\reddit_test.csv
Train Shape: (30068, 56)
Test Shape : (7517, 56)

PROCESSING: Facebook
Original Shape: (7527, 32)

Final ML Sh